In [ ]:
import colorcet as cc
import gstools as gs
import jax
import matplotlib.pyplot as plt
import numpy as np
from context_flux_no.simulations.pde.euler import Euler2D
from context_flux_no.simulations.utils import generate_dataset
from context_flux_no.waveforms.grf import GaussianRandomField
from context_flux_no.waveforms.step import PeriodicRandomStepFunction2D

In [ ]:
grf = GaussianRandomField(
    covariance_fns=[
        gs.Gaussian(dim=2, len_scale=0.2),
        gs.Gaussian(dim=2, len_scale=0.2),
        gs.Gaussian(dim=2, len_scale=0.2),
        gs.Gaussian(dim=2, len_scale=0.2),
    ],
    means=[1, 0, 0, 1],
    scales=[0.2, 0.6, 0.6, 0.2],
)

In [ ]:
u_test = grf.sample((np.linspace(0, 1, 128), np.linspace(0, 1, 128)), jax.random.key(0))

In [ ]:
u_test.shape

In [ ]:
gamma = 1.4
mach = np.hypot(u_test[1], u_test[2]) / np.sqrt(gamma * u_test[3] / u_test[0])

fig, axes = plt.subplots(1, 5, figsize=(10, 3))
for i in range(u_test.shape[0]):
    axes[i].hist(u_test[i].flatten(), bins=100)
axes[-1].hist(mach.flatten(), bins=100)

In [ ]:
euler_claw = Euler2D(gamma=1.5)
sol_claw = euler_claw.solve(
    lambda xs: grf.sample(xs, key=jax.random.key(10)),
    ((0.0, 1.0), (0.0, 1.0)),
    (128, 128),
    (0.0, 0.5),
    100,
    "periodic",
    use_rho_v_p_ics=True,
)

In [ ]:
sol_claw[0].shape

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(sol_claw[0][i * 10, 0], cmap=cc.cm.coolwarm)

In [ ]:
dataset = generate_dataset(
    dataset_name="2d_euler_test",
    n_coeffs=10,
    n_ics_per_coeff=10,
    pde_factory=Euler2D,
    initial_condition_fn=grf.sample,
    coeff_range_dict={"gamma": (1.2, 1.5)},
    x_spans=((0, 1), (0, 1)),
    Nxs=(128, 128),
    t_span=(0, 0.5),
    Nt=100,
    coeff_sampling_strategy="linspace",
    seed=0,
)

In [ ]:
dataset[10]

In [ ]:
await dataset.root.store.getsize_prefix("") / 1e9

In [ ]:
dataset.root.tree()

## OOD dataset

In [ ]:
step_2d = PeriodicRandomStepFunction2D(
    channels=4,
    num_regions_min=2,
    num_regions_max=8,
    value_min=[0.6, -0.8, -0.8, 0.6],
    value_max=[1.4, 0.8, 0.8, 1.4],
)

In [ ]:
u_test = step_2d.sample(
    (np.linspace(0, 1, 128), np.linspace(0, 1, 128)), jax.random.key(0)
)

In [ ]:
u_test.shape

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(8, 4))
for i, ax in enumerate(axes):
    ax.imshow(u_test[i])

In [ ]:
euler_claw = Euler2D(gamma=1.5)
sol_claw = euler_claw.solve(
    lambda xs: step_2d.sample(xs, key=jax.random.key(50)),
    ((0.0, 1.0), (0.0, 1.0)),
    (128, 128),
    (0.0, 0.5),
    100,
    "periodic",
    use_rho_v_p_ics=True,
)

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(sol_claw[0][i * 10, 0], cmap=cc.cm.coolwarm, vmin=0.2, vmax=1.8)

In [ ]:
dataset = generate_dataset(
    dataset_name="2d_euler_test",
    n_coeffs=10,
    n_ics_per_coeff=10,
    pde_factory=Euler2D,
    initial_condition_fn=step_2d.sample,
    coeff_range_dict={"gamma": (1.2, 1.5)},
    x_spans=((0, 1), (0, 1)),
    Nxs=(128, 128),
    t_span=(0, 0.5),
    Nt=100,
    coeff_sampling_strategy="linspace",
    seed=0,
    use_rho_v_p_ics=True,
)

In [ ]:
len(dataset)